# BENCHMARK TOÀN DIỆN MÔ HÌNH CLIP TRÊN PHÂN LOẠI GIỐNG ĐỘNG VẬT
## NewDataset (N Classes) - Đối Sánh Toàn Diện CLIP Zero-Shot, Tip-Adapter (Free), Tip-Adapter-F (Tuned), CLIP-Adapter & Prompt Tuning (CoOp)

---

### 1. Bối Cảnh & Cơ Sở Lý Thuyết
Mô hình nền tảng **CLIP (Contrastive Language-Image Pre-training)** sở hữu khả năng biểu diễn đa thức không gian thị giác - ngôn ngữ mạnh mẽ. Tuy nhiên, khi áp dụng trực tiếp cho bài toán phân loại giống động vật chi tiết (Fine-grained Pet Breed Classification với N giống), mô hình Zero-Shot có thể gặp hạn chế đối với các giống có ngoại hình tương đồng.

Notebook này tiến hành **Benchmark toàn diện** 5 phương pháp thích ứng tiên tiến trên mô hình CLIP (`openai/clip-vit-base-patch32`):
1. **CLIP Baseline (Zero-Shot Classification)**: Đánh giá khả năng nhận diện trực tiếp bằng Enriched Prompt Engineering.
2. **CLIP + Tip-Adapter (Training-Free Cache-Model)**: Bộ nhớ đệm cache không cần huấn luyện, tìm kiếm siêu tham số $\alpha, \beta$ bằng Grid Search.
3. **CLIP + Tip-Adapter-F (Fine-Tuned Cache-Model)**: Tinh chỉnh lan truyền ngược ma trận Key cache $W_1$ với các tham số thặng dư.
4. **CLIP + CLIP-Adapter (Residual Feature Adapter)**: Tích hợp mạng nơ-ron MLP Bottleneck thặng dư ($\alpha = 0.2$).
5. **CLIP + Prompt Tuning (CoOp - Context Optimization)**: Tối ưu hóa 4 vector ngữ cảnh liên tục ($2,048$ tham số) trong không gian embedding của Text Encoder.

---

# BENCHMARK TOÀN DIỆN CLIP — NEWDATASET

Chỉ huấn luyện/đánh giá các thư mục lớp trong `NewDataset`; không gộp Oxford-IIIT Pet.
Notebook độc lập, giữ thứ tự và các phương pháp từ `benchmark_clip_animal_classifier.ipynb`.

**Kaggle:** Import notebook này; Add Input dataset chứa `NewDataset/<giống>/<ảnh>`; bật GPU và Internet.
**MacBook:** cài dependencies ở cell dưới, chọn kernel Python của môi trường; tự chọn CUDA → MPS → CPU.
Đường dẫn `DATA_ROOT` phải trỏ tới thư mục chứa trực tiếp các thư mục giống.

**Thiết kế:** loại ảnh trùng pixel trước khi chia, giữ 20% mỗi lớp làm test; 80% còn lại là trainval.
Trainval chia 9:1 theo lớp. Full trainval dùng toàn bộ 80%, không dùng test để chọn tham số.
Ảnh gần trùng hoặc cùng một cá thể không được phát hiện bởi hash; cần kiểm tra nguồn dữ liệu nếu muốn split theo cá thể.
Cache tách theo fingerprint dữ liệu và backbone; mọi output ghi vào thư mục có quyền ghi.
Các số liệu và biểu đồ sẽ được tính khi chạy, không sao chép kết quả Oxford cũ.


In [ ]:
%pip install "transformers>=4.45,<5" "opencv-python-headless>=4.8,<5" scikit-learn matplotlib seaborn pandas tabulate tqdm
# macOS only, if torch/torchvision are missing: %pip install torch torchvision


In [ ]:
import os
import time
import json
import random
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision
from torchvision.datasets import OxfordIIITPet
from transformers import CLIPModel, CLIPProcessor

# Cấu hình hạt giống tái lập (Reproducibility)
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)

# Thiết lập thiết bị tính toán tối ưu (MPS cho Mac Apple Silicon / CUDA / CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"🚀 Thiết bị tính toán được kích hoạt: {device}")

# Cấu hình đồ họa Matplotlib
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#D0D5DD'
plt.rcParams['axes.linewidth'] = 1.0
sns.set_theme(style="whitegrid")

RUN_NAME = 'clip'

from pathlib import Path
def as_feature_tensor(value):
    # Transformers 4 returns a tensor; newer APIs may return a pooled model output.
    return value if isinstance(value, torch.Tensor) else value.pooler_output

DATA_ROOT = None  # e.g. "/kaggle/input/animal-newdataset/NewDataset"
SEED = 42
TEST_RATIO = 0.20
BATCH_SIZE = 128  # original benchmark; reduce to 32/16/8 if memory is limited
TIP_EPOCHS = 20
ADAPTER_EPOCHS = 50
COOP_EPOCHS = 50
RESNET_EPOCHS = 30
OUTPUT_ROOT = Path('/kaggle/working/newdataset_benchmark') if Path('/kaggle').exists() else Path('results/newdataset_benchmark')
OUTPUT_ROOT = OUTPUT_ROOT.resolve()
output_dir = OUTPUT_ROOT / RUN_NAME
output_dir.mkdir(parents=True, exist_ok=True)
set_seed(SEED)
"""Runtime reports: measured results only, with explicit latency scope."""
def synchronize():
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elif device.type == 'mps':
        torch.mps.synchronize()


def measure_latency_ms(fn, num_runs=50, warmup=5):
    with torch.no_grad():
        for _ in range(warmup):
            fn()
        synchronize()
        times = []
        for _ in range(num_runs):
            synchronize()
            start = time.perf_counter()
            fn()
            synchronize()
            times.append((time.perf_counter() - start) * 1000)
    return {key: float(value) for key, value in {
        'mean': np.mean(times), 'median': np.median(times),
        'p95': np.percentile(times, 95), 'p99': np.percentile(times, 99)}.items()}


def save_checkpoint(name, model, **extra):
    # CoOp saves only the learned context; pretrained CLIP is shared and frozen.
    state = {'ctx': model.ctx.detach().cpu()} if hasattr(model, 'ctx') else {
        key: value.detach().cpu() for key, value in model.state_dict().items()}
    torch.save({'state_dict': state, 'classes': class_names,
                'fingerprint': manifest['fingerprint'], 'seed': SEED,
                'backbone': MODEL_ID,
                'alpha': getattr(model, 'alpha', None), 'beta': getattr(model, 'beta', None),
                'logit_scale': globals().get('logit_scale_val'), **extra}, output_dir / f'{name}.pt')


def measure_scenarios(scenario_predictors):
    global scenario_latency
    scenarios = ['Raw Image (Full Trainval)', 'TTA 5-view (9:1 Split)', 'TTA 5-view (Full Trainval)']
    scenario_latency = {}
    for method, heads in scenario_predictors.items():
        scenario_latency[method] = []
        for scenario, head in zip(scenarios, heads):
            sample = (test_features_raw if scenario.startswith('Raw') else test_features_tta)[:1].to(device)
            stats = measure_latency_ms(lambda: head(sample))
            scenario_latency[method].append(stats['mean'])
            for row in results.values():
                if row['Method'] == method and row['Scenario'] == scenario:
                    row['Head_Latency_ms'] = stats['mean']
                    row['Head_Throughput_Samples_Sec'] = 1000 / stats['mean']
                    row['Head_Latency_Stats'] = stats


def pareto_frontier(points):
    best, frontier = -float('inf'), []
    for latency, accuracy in sorted((float(x), float(y)) for x, y in points if np.isfinite(x) and np.isfinite(y)):
        if accuracy > best:
            frontier.append((latency, accuracy)); best = accuracy
    return ([p[0] for p in frontier], [p[1] for p in frontier])


def matching_clip_results():
    path = output_dir.parent / 'clip' / 'benchmark_results.json'
    if not path.exists():
        return None
    data = json.loads(path.read_text(encoding='utf-8'))
    return data if data['dataset'] == manifest['fingerprint'] else None


def load_matching_clip_scores():
    data = matching_clip_results()
    values = {'Raw': float('nan'), 'TTA_90': float('nan'), 'TTA_Full': float('nan')}
    scenarios = dict(zip(['Raw Image (Full Trainval)', 'TTA 5-view (9:1 Split)', 'TTA 5-view (Full Trainval)'], values))
    if data:
        for row in data['results']:
            if row['Method'] == 'CLIP + Prompt Tuning':
                values[scenarios[row['Scenario']]] = row['Test_Acc']
    else:
        print('CLIP comparison pending: run CLIP on the same dataset split and output root, then rerun this report cell.')
    return values


def matching_clip_latency(scenario):
    data = matching_clip_results()
    if data and data['device'] == str(device):
        for row in data['results']:
            if row['Method'] == 'CLIP + Prompt Tuning' and row['Scenario'] == scenario:
                return row.get('Head_Latency_ms', float('nan'))
    return float('nan')


def build_report(results, predictors, histories, encoder):
    import pandas as pd
    from IPython.display import display
    frame = pd.DataFrame(results.values()).drop(columns=['Head_Latency_Stats'], errors='ignore')
    frame.insert(0, 'STT', range(1, len(frame) + 1))
    frame['Preprocessing'] = np.where(frame.Scenario.str.startswith('Raw'), 'Raw Images', 'TTA 5 Views')
    base = float(frame.iloc[0]['Test_Acc'])
    frame['Delta_vs_Baseline'] = frame['Test_Acc'] - base
    latency = {key: row['Head_Latency_Stats'] for key, row in results.items()}
    # Encoder timing uses synthetic pixels already on device; excludes PIL, OpenCV and IO.
    pixels = torch.zeros(1, 3, 224, 224, device=device)
    latency['encoder_raw'] = measure_latency_ms(lambda: encoder(pixels))
    latency['encoder_5_views'] = measure_latency_ms(lambda: encoder(pixels.expand(5, -1, -1, -1)))
    frame.to_csv(output_dir / 'benchmark_results.csv', index=False)
    display(frame)
    summary = {'dataset': manifest['fingerprint'], 'classes': class_names,
               'device': str(device), 'backbone': MODEL_ID, 'seed': SEED,
               'latency_scope': 'Each scenario head on cached raw/TTA features; encoder measured separately on device-resident synthetic pixels. Not end-to-end.',
               'latency_ms': latency, 'results': list(results.values())}
    (output_dir / 'benchmark_results.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
    (output_dir / 'training_history.json').write_text(json.dumps(histories, indent=2), encoding='utf-8')
    (output_dir / 'environment.json').write_text(json.dumps({
        'torch': torch.__version__, 'torchvision': torchvision.__version__,
        'transformers': __import__('transformers').__version__, 'device': str(device),
        'gpu': torch.cuda.get_device_name() if device.type == 'cuda' else None}, indent=2), encoding='utf-8')

    # Match the original report: CLIP CoOp / ResNet Cosine, both 9:1 + TTA.
    selected = 'CLIP + Prompt Tuning' if output_dir.name == 'clip' else 'ResNet-50 (Cosine Head)'
    with torch.no_grad():
        preds = predictors[selected](test_features_tta.to(device)).argmax(-1).cpu().numpy()
    truth = test_labels.cpu().numpy()
    matrix = confusion_matrix(truth, preds, labels=list(range(num_classes)))
    report = classification_report(truth, preds, target_names=class_names, output_dict=True, zero_division=0)
    (output_dir / 'classification_report.json').write_text(json.dumps({
        'reported_method': selected, 'report': report,
        'confusion_matrix': matrix.tolist()}, indent=2), encoding='utf-8')
    rows = frame.fillna('—').astype(str)
    table = '| ' + ' | '.join(rows.columns) + ' |\n'
    table += '| ' + ' | '.join(['---'] * len(rows.columns)) + ' |\n'
    table += '\n'.join('| ' + ' | '.join(row) + ' |' for row in rows.to_numpy())
    (output_dir / 'benchmark_report.md').write_text(
        f'# {MODEL_ID} — NewDataset\n\n{table}\n\n'
        'Latency: head-only and encoder-only, excluding image IO and preprocessing. '
        'Full trainval has no independent validation score.\n', encoding='utf-8')
    # Compare only with a matching split fingerprint, never legacy Oxford scores.
    sibling = output_dir.parent / ('resnet50' if output_dir.name == 'clip' else 'clip') / 'benchmark_results.json'
    if sibling.exists():
        other = json.loads(sibling.read_text(encoding='utf-8'))
        if other['dataset'] == manifest['fingerprint']:
            combined = pd.concat([frame.assign(Backbone=MODEL_ID),
                                  pd.DataFrame(other['results']).assign(Backbone=other['backbone'])])
            combined.to_csv(output_dir / 'clip_vs_resnet50.csv', index=False)
            display(combined[['Backbone', 'Method', 'Scenario', 'Val_Acc', 'Test_Acc']])
        else:
            print('Skipped comparison: different split fingerprints')
    print(f'Artifacts saved to {output_dir}')


In [ ]:
"""Image-folder dataset, content deduplication and reproducible benchmark splits."""
import hashlib
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

from PIL import Image, ImageOps
from torch.utils.data import Dataset

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}


def discover_dataset(explicit=None):
    if explicit:
        root = Path(explicit).expanduser()
        if not root.is_dir():
            raise FileNotFoundError(root)
        return root.resolve()
    candidates = []
    for parent in [Path.cwd(), *Path.cwd().parents]:
        for name in ('NewDataset', 'NewDatasett'):
            candidate = parent / 'dataset' / name
            if candidate.is_dir():
                candidates.append(candidate.resolve())
    kaggle = Path('/kaggle/input')
    if kaggle.exists():
        candidates.extend(p.resolve() for p in kaggle.rglob('*')
                          if p.is_dir() and p.name in ('NewDataset', 'NewDatasett'))
    candidates = sorted(set(candidates))
    if len(candidates) != 1:
        raise ValueError(f'Set DATA_ROOT to the folder containing breed folders; found {candidates}')
    return candidates[0]


class BreedDataset(Dataset):
    def __init__(self, root, records, classes):
        self.root, self.records, self.classes = Path(root), records, classes

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        item = self.records[index]
        with Image.open(self.root / item['path']) as image:
            image = ImageOps.exif_transpose(image).convert('RGB')
        return image, item['label']


def prepare_dataset(root, output_dir, seed=42, test_ratio=0.2):
    """Split unique decoded images per class; reject conflicting duplicate labels."""
    root, output_dir = Path(root), Path(output_dir)
    if not 0 < test_ratio < 1:
        raise ValueError('test_ratio must be between 0 and 1')
    classes = sorted(p.name for p in root.iterdir() if p.is_dir() and not p.name.startswith('.'))
    if len(classes) < 2:
        raise ValueError('Expected at least two breed folders')
    groups, seen, duplicates, invalid = defaultdict(list), {}, [], []
    for label, name in enumerate(classes):
        for path in sorted((root / name).rglob('*')):
            if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            relative = path.relative_to(root).as_posix()
            try:
                with Image.open(path) as image:
                    image = ImageOps.exif_transpose(image).convert('RGB')
                    digest = hashlib.sha256(str(image.size).encode() + image.tobytes()).hexdigest()
            except (OSError, ValueError) as error:
                invalid.append({'path': relative, 'error': str(error)})
                continue
            if digest in seen:
                previous = seen[digest]
                if previous['label'] != label:
                    raise ValueError(f'Identical image has conflicting labels: {previous["path"]}, {relative}')
                duplicates.append({'path': relative, 'same_as': previous['path']})
                continue
            record = {'path': relative, 'label': label, 'sha256': digest}
            seen[digest] = record
            groups[label].append(record)
    trainval, test = [], []
    for label, name in enumerate(classes):
        items = groups[label][:]
        if len(items) < 4:
            raise ValueError(f'{name}: need >= 4 valid unique images for train/val/test')
        random.Random(seed + label).shuffle(items)
        count = min(len(items) - 2, max(1, round(len(items) * test_ratio)))
        test.extend(items[:count])
        trainval.extend(items[count:])
    manifest = {'seed': seed, 'test_ratio': test_ratio, 'classes': classes,
                'trainval': trainval, 'test': test, 'duplicates_removed': duplicates,
                'invalid_images': invalid, 'version': 1}
    fingerprint = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()[:16]
    manifest['fingerprint'] = fingerprint
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / 'split_manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
    (output_dir / 'classes.json').write_text(json.dumps(classes, indent=2), encoding='utf-8')
    print(f'{len(classes)} classes | trainval={len(trainval)} | test={len(test)}')
    print(f'Removed duplicates={len(duplicates)}; invalid images={len(invalid)}; fingerprint={fingerprint}')
    print({classes[k]: v for k, v in sorted(Counter(r['label'] for r in trainval + test).items())})
    return BreedDataset(root, trainval, classes), BreedDataset(root, test, classes), manifest

dataset_root = discover_dataset(DATA_ROOT)
pets_trainval, pets_test, manifest = prepare_dataset(dataset_root, output_dir, SEED, TEST_RATIO)
class_names = pets_trainval.classes
prompt_class_names = [name.replace('_', ' ') for name in class_names]
num_classes = len(class_names)
trainval_labels = torch.tensor([r['label'] for r in pets_trainval.records], dtype=torch.long)
test_labels = torch.tensor([r['label'] for r in pets_test.records], dtype=torch.long)

MODEL_ID = 'openai/clip-vit-base-patch32'
clip_model = CLIPModel.from_pretrained(MODEL_ID).to(device).eval()
clip_model.requires_grad_(False)
clip_processor = CLIPProcessor.from_pretrained(MODEL_ID)
logit_scale_val = clip_model.logit_scale.exp().item()
prompts_raw = [c.replace('_', ' ') for c in class_names]
prompts_enriched = [f'Picture of a pet with breed: {c.replace("_", " ")}, which is a type of animal.' for c in class_names]


## 1. QUY TRÌNH TIỀN XỬ LÝ DỮ LIỆU TỐI ƯU (OPTIMAL PREPROCESSING PIPELINE) & TTA

---

Kế thừa từ quy trình tiền xử lý ảnh tối ưu:
1. **Khử Nhiễu Kết Cấu Lông (Bilateral Filter)**: Giữ nguyên biên cạnh đặc trưng của mắt và tai thú cưng.
2. **Cân Bằng Phổ Sáng Tự Nhiên (CLAHE trên kênh L không gian màu LAB)**: Làm nổi bật chi tiết lông mờ trong điều kiện thiếu sáng hoặc ngược sáng.
3. **TTA 5-view (Test-Time Augmentation)**: Tổng hợp 5 góc nhìn biến đổi hình học (Original, Denoise, CLAHE, Combined, Combined + Flip Horizontal + Rotate 15°) tại thời điểm suy diễn.

In [ ]:
# Hiện thực các hàm tiền xử lý hình ảnh
def apply_denoise(pil_img):
    img_bgr = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    denoised_bgr = cv2.bilateralFilter(img_bgr, d=7, sigmaColor=50, sigmaSpace=50)
    return Image.fromarray(cv2.cvtColor(denoised_bgr, cv2.COLOR_BGR2RGB))

def apply_clahe(pil_img):
    img_bgr = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    clahe_bgr = cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2BGR)
    return Image.fromarray(cv2.cvtColor(clahe_bgr, cv2.COLOR_BGR2RGB))

def optimal_preprocessing_pipeline(pil_img):
    img_bgr = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    denoised_bgr = cv2.bilateralFilter(img_bgr, d=7, sigmaColor=50, sigmaSpace=50)
    lab = cv2.cvtColor(denoised_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    clahe_bgr = cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2BGR)
    return Image.fromarray(cv2.cvtColor(clahe_bgr, cv2.COLOR_BGR2RGB))

transform_geom = lambda img: optimal_preprocessing_pipeline(img).transpose(Image.Transpose.FLIP_LEFT_RIGHT).rotate(15)

class TransformedDataset(Dataset):
    def __init__(self, base_dataset, transform_fn, clip_processor):
        self.base_dataset = base_dataset
        self.transform_fn = transform_fn
        self.clip_processor = clip_processor
        
    def __len__(self):
        return len(self.base_dataset)
        
    def __getitem__(self, idx):
        img, label = self.base_dataset[idx]
        if self.transform_fn is not None:
            img = self.transform_fn(img)
        pixel_values = self.clip_processor(images=img, return_tensors="pt")["pixel_values"][0]
        return pixel_values, label

def extract_features(dataset, batch_size=128, desc="Trích xuất đặc trưng"):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    feats = []
    with torch.no_grad():
        for batch_pixels, _ in tqdm(loader, desc=desc):
            batch_pixels = batch_pixels.to(device)
            f = as_feature_tensor(clip_model.get_image_features(pixel_values=batch_pixels))
            f = f / f.norm(dim=-1, keepdim=True)
            feats.append(f.cpu())
    return torch.cat(feats, dim=0)


cache_dir = output_dir / 'cache' / manifest['fingerprint'] / 'clip_vit_b32_preprocess_v1'
cache_dir.mkdir(parents=True, exist_ok=True)
def get_features(name, dataset, transform=None):
    path = cache_dir / f'{name}.pt'
    if path.exists():
        features = torch.load(path, map_location='cpu', weights_only=True)
        assert features.shape == (len(dataset), clip_model.config.projection_dim)
        return features
    features = extract_features(TransformedDataset(dataset, transform, clip_processor), batch_size=BATCH_SIZE, desc=name)
    torch.save(features, path)
    return features
trainval_features_raw = get_features('trainval_raw', pets_trainval)
test_features_raw = get_features('test_raw', pets_test)
test_features_opt = get_features('test_optimal', pets_test, optimal_preprocessing_pipeline)
test_features_tta = F.normalize(torch.stack([
    test_features_raw, get_features('test_denoise', pets_test, apply_denoise),
    get_features('test_clahe', pets_test, apply_clahe), test_features_opt,
    get_features('test_geom', pets_test, transform_geom)]).mean(0), dim=-1)
print(trainval_features_raw.shape, test_features_tta.shape)


In [ ]:
# Trực quan hóa ảnh mẫu và các góc nhìn trong quy trình tiền xử lý tối ưu
sample_idx = min(10, len(pets_test) - 1)
sample_img, sample_lbl = pets_test[sample_idx]

view_orig = sample_img
view_denoise = apply_denoise(sample_img)
view_clahe = apply_clahe(sample_img)
view_optimal = optimal_preprocessing_pipeline(sample_img)
view_geom = transform_geom(sample_img)

fig, axes = plt.subplots(1, 5, figsize=(20, 4.5))
views_info = [
    (view_orig, f"1. Gốc (Original)\nGiống: {class_names[sample_lbl]}"),
    (view_denoise, "2. Bilateral Denoise\n(Khử nhiễu, giữ biên lông)"),
    (view_clahe, "3. CLAHE LAB\n(Tăng tương phản sáng)"),
    (view_optimal, "4. Optimal Combined\n(Bilateral + CLAHE LAB)"),
    (view_geom, "5. Geometric TTA\n(Optimal + Flip + Rotate 15°)")
]

for ax, (v_img, title) in zip(axes, views_info):
    ax.imshow(v_img)
    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
    ax.axis("off")

plt.suptitle("QUY TRÌNH TIỀN XỬ LÝ & CÁC GÓC NHÌN TTA ĐA CHIỀU TRÊN NEWDATASET", fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()


## 2. CHIẾN LƯỢC PHÂN CHIA DỮ LIỆU: TRAIN:VAL (9:1) & FULL TRAINVAL

20% ảnh duy nhất mỗi lớp được giữ làm test từ đầu. Chia 80% trainval còn lại theo lớp:
90% để train/support, 10% validation chọn tham số và checkpoint có validation accuracy cao nhất.
Full trainval không có validation độc lập; chạy đúng số epochs đã cấu hình.
Đây là chọn checkpoint tốt nhất, không phải dừng sớm. CLIP train trên raw features;
ResNet có cả raw và TTA train features như benchmark gốc.


In [ ]:

# Identical raw/TTA and CLIP/ResNet split indices; at least one validation sample/class.
train_indices, val_indices = [], []
set_seed(SEED)  # preserve the original global RNG sequence before fitting heads
for c in range(num_classes):
    indices = (trainval_labels == c).nonzero(as_tuple=True)[0]
    indices = indices[torch.randperm(len(indices))]
    n_train = min(len(indices) - 1, max(1, int(len(indices) * 0.9)))
    train_indices.append(indices[:n_train]); val_indices.append(indices[n_train:])
train_idx, val_idx = torch.cat(train_indices), torch.cat(val_indices)
(output_dir / 'train_val_indices.json').write_text(json.dumps({
    'train': train_idx.tolist(), 'val': val_idx.tolist(), 'fingerprint': manifest['fingerprint']}, indent=2), encoding='utf-8')
print(f'Train={len(train_idx)} | Validation={len(val_idx)} | Test={len(test_labels)}')

train_feats_90, train_lbls_90 = trainval_features_raw[train_idx], trainval_labels[train_idx]
val_feats_10, val_lbls_10 = trainval_features_raw[val_idx], trainval_labels[val_idx]
train_feats_full, train_lbls_full = trainval_features_raw, trainval_labels
benchmark_12_results = {}


## 3. PHƯƠNG PHÁP 1: CLIP BASELINE (ZERO-SHOT CLASSIFICATION)

---

Mô hình CLIP thực hiện phân loại Zero-Shot bằng cách tính độ tương đồng Cosine giữa vector đặc trưng hình ảnh $\mathbf{f}_{\text{img}}$ và ma trận đặc trưng câu lệnh văn bản $\mathbf{W}_{\text{text}}$:
$$\text{Logits} = \tau \cdot \left(\mathbf{f}_{\text{img}} \mathbf{W}_{\text{text}}^T\right)$$
*(với $\tau = 100.0$ là logit scale chuẩn của CLIP)*.

Chúng ta so sánh 2 kỹ thuật Prompt:
- **Bare Class Names**: `"Abyssinian"`, `"Beagle"`, ...
- **Enriched Prompt Engineering**: `"Picture of a pet with breed: {c}, which is a type of animal."`

In [ ]:
# Trích xuất đặc trưng văn bản Enriched
with torch.no_grad():
    inputs_enr = clip_processor(text=prompts_enriched, return_tensors="pt", padding=True).to(device)
    text_feats_enr = as_feature_tensor(clip_model.get_text_features(**inputs_enr))
    text_feats_enr = text_feats_enr / text_feats_enr.norm(dim=-1, keepdim=True)
    
    inputs_raw = clip_processor(text=prompts_raw, return_tensors="pt", padding=True).to(device)
    text_feats_raw = as_feature_tensor(clip_model.get_text_features(**inputs_raw))
    text_feats_raw = text_feats_raw / text_feats_raw.norm(dim=-1, keepdim=True)

# 1.1. Raw Test
sim_raw = test_features_raw.to(device) @ text_feats_enr.to(device).t()
preds_base_raw = sim_raw.argmax(dim=-1)
acc_base_raw = (preds_base_raw == test_labels.to(device)).float().mean().item() * 100

# 1.2 & 1.3. TTA 5-views Test
sim_tta = test_features_tta.to(device) @ text_feats_enr.to(device).t()
preds_base_tta = sim_tta.argmax(dim=-1)
acc_base_tta_90 = (preds_base_tta == test_labels.to(device)).float().mean().item() * 100
acc_base_tta_full = acc_base_tta_90

# Zero-shot raw labels baseline reference
acc_raw_labels = ((test_features_raw.to(device) @ text_feats_raw.to(device).t()).argmax(dim=-1) == test_labels.to(device)).float().mean().item() * 100

benchmark_12_results["1. CLIP (Baseline) | Raw Image (Full Trainval)"] = {
    "Method": "CLIP (Baseline)", "Scenario": "Raw Image (Full Trainval)", "Val_Acc": None, "Test_Acc": acc_base_raw, "Trainable_Params": 0
}
benchmark_12_results["1. CLIP (Baseline) | TTA 5-view (9:1 Split)"] = {
    "Method": "CLIP (Baseline)", "Scenario": "TTA 5-view (9:1 Split)", "Val_Acc": None, "Test_Acc": acc_base_tta_90, "Trainable_Params": 0
}
benchmark_12_results["1. CLIP (Baseline) | TTA 5-view (Full Trainval)"] = {
    "Method": "CLIP (Baseline)", "Scenario": "TTA 5-view (Full Trainval)", "Val_Acc": None, "Test_Acc": acc_base_tta_full, "Trainable_Params": 0
}

print("="*75)
print("🏆 KẾT QUẢ ĐÁNH GIÁ PHƯƠNG PHÁP 1: CLIP BASELINE")
print("="*75)
print(f"  [Ref] Raw Class Names Zero-Shot:          {acc_raw_labels:.2f}%")
print(f"  1.1. Raw Image (Full Trainval / ZS):      {acc_base_raw:.2f}%")
print(f"  1.2. TTA 5-view (9:1 Split):             {acc_base_tta_90:.2f}%")
print(f"  1.3. TTA 5-view (Full Trainval):         {acc_base_tta_full:.2f}%")
print("="*75)


## 4. PHƯƠNG PHÁP 2 & 3: CLIP + TIP-ADAPTER (FREE & TUNED TIP-ADAPTER-F)

---

**Tip-Adapter** kế thừa kiến trúc bộ nhớ đệm Cache-Model (Key-Value Cache) được xây dựng từ đặc trưng tập huấn luyện hỗ trợ ($\mathbf{F}_{\text{train}}, \mathbf{L}_{\text{train}}$):
- **Tip-Adapter (Training-Free)**: Không cần lan truyền ngược (0 tham số huấn luyện). Tối ưu hai siêu tham số $\alpha$ (tỷ lệ hòa trộn thặng dư) và $\beta$ (độ sắc nhọn hàm Exponential Sharpness) bằng thuật toán Grid Search trên tập Validation 10%:
  $$\mathbf{A} = \exp\left(-\beta (1 - \mathbf{f}_{\text{test}} \mathbf{W}_1^T)\right)$$
  $$\text{Logits}_{\text{Free}} = \tau \cdot (\mathbf{f}_{\text{test}} \mathbf{W}_{\text{text}}^T) + \alpha \frac{\mathbf{A} \mathbf{L}_{\text{values}}}{K_{\text{avg}}}$$
- **Tip-Adapter-F (Fine-Tuned / Adaptive Keys)**: Khởi tạo trọng số Key Cache $\mathbf{W}_1$ từ các vector đặc trưng hỗ trợ, sau đó tinh chỉnh lan truyền ngược bằng thuật toán SGD qua 20 epochs nhằm tối ưu hóa khả năng thích ứng của các vector chìa khóa theo miền dữ liệu thú cưng.

Chúng ta tiến hành đối sánh đồng thời cả 2 phương pháp trên 3 kịch bản dữ liệu:
- **Raw Image (Full Trainval)**
- **TTA 5-view (9:1 Split)**
- **TTA 5-view (Full Trainval)**

In [ ]:
# ==============================================================================
# PHƯƠNG PHÁP 2: TIP-ADAPTER (TRAINING-FREE)
# ==============================================================================
def run_tip_adapter_free(test_f, text_f, supp_f, supp_l, alpha=1.0, beta=10.0):
    w1_norm = supp_f / supp_f.norm(dim=-1, keepdim=True)
    w2 = F.one_hot(supp_l, num_classes=num_classes).float()
    logits_zs = (test_f @ text_f.t()) * logit_scale_val
    sim_matrix = test_f @ w1_norm.t()
    A = torch.exp(-beta * (1.0 - sim_matrix))
    K_avg = len(supp_f) / float(num_classes)
    logits_cache = ((A @ w2) / K_avg) * logit_scale_val
    return logits_zs + alpha * logits_cache

def grid_search_tip_free(val_f, val_l, text_f, supp_f, supp_l):
    alphas = [0.1, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0]
    betas = [1.0, 2.0, 3.0, 5.0, 10.0]
    best_acc, best_a, best_b = 0.0, 1.0, 5.0
    for a in alphas:
        for b in betas:
            logits = run_tip_adapter_free(val_f.to(device), text_f.to(device), supp_f.to(device), supp_l.to(device), a, b)
            acc = (logits.argmax(dim=-1) == val_l.to(device)).float().mean().item() * 100.0
            if acc > best_acc:
                best_acc, best_a, best_b = acc, a, b
    return best_a, best_b, best_acc

print("🔍 Đang thực hiện Grid Search tối ưu siêu tham số alpha & beta cho Tip-Adapter (Free)...")
best_alpha_90, best_beta_90, val_acc_tip_free_90 = grid_search_tip_free(
    val_feats_10, val_lbls_10, text_feats_enr, train_feats_90, train_lbls_90
)
print(f"✅ Grid Search hoàn tất trên tập Val 10%: Best Alpha = {best_alpha_90}, Best Beta = {best_beta_90} (Val Acc: {val_acc_tip_free_90:.2f}%)")

# Đánh giá Tip-Adapter (Free) trên 3 kịch bản
acc_tip_free_raw = (run_tip_adapter_free(test_features_raw.to(device), text_feats_enr.to(device), train_feats_full.to(device), train_lbls_full.to(device), best_alpha_90, best_beta_90).argmax(dim=-1) == test_labels.to(device)).float().mean().item() * 100
acc_tip_free_90_tta = (run_tip_adapter_free(test_features_tta.to(device), text_feats_enr.to(device), train_feats_90.to(device), train_lbls_90.to(device), best_alpha_90, best_beta_90).argmax(dim=-1) == test_labels.to(device)).float().mean().item() * 100
acc_tip_free_full_tta = (run_tip_adapter_free(test_features_tta.to(device), text_feats_enr.to(device), train_feats_full.to(device), train_lbls_full.to(device), best_alpha_90, best_beta_90).argmax(dim=-1) == test_labels.to(device)).float().mean().item() * 100

benchmark_12_results["2. CLIP + Tip-Adapter (Free) | Raw Image (Full Trainval)"] = {
    "Method": "CLIP + Tip-Adapter (Free)", "Scenario": "Raw Image (Full Trainval)", "Val_Acc": None, "Test_Acc": acc_tip_free_raw, "Trainable_Params": 0
}
benchmark_12_results["2. CLIP + Tip-Adapter (Free) | TTA 5-view (9:1 Split)"] = {
    "Method": "CLIP + Tip-Adapter (Free)", "Scenario": "TTA 5-view (9:1 Split)", "Val_Acc": val_acc_tip_free_90, "Test_Acc": acc_tip_free_90_tta, "Trainable_Params": 0
}
benchmark_12_results["2. CLIP + Tip-Adapter (Free) | TTA 5-view (Full Trainval)"] = {
    "Method": "CLIP + Tip-Adapter (Free)", "Scenario": "TTA 5-view (Full Trainval)", "Val_Acc": None, "Test_Acc": acc_tip_free_full_tta, "Trainable_Params": 0
}

# ==============================================================================
# PHƯƠNG PHÁP 3: TIP-ADAPTER-F (FINE-TUNED)
# ==============================================================================
class TipAdapterF(nn.Module):
    def __init__(self, support_features, support_labels, text_features, alpha=1.0, beta=10.0):
        super().__init__()
        self.W_1 = nn.Parameter(support_features.clone())
        self.register_buffer("W_2", F.one_hot(support_labels, num_classes=num_classes).float())
        self.register_buffer("W_text", text_features.clone())
        self.alpha = alpha
        self.beta = beta
        self.register_buffer("K_avg", torch.tensor(len(support_features) / float(num_classes)))
        
    def forward(self, x):
        logits_zs = (x @ self.W_text.t()) * logit_scale_val
        w1_norm = self.W_1 / self.W_1.norm(dim=-1, keepdim=True)
        sim_matrix = x @ w1_norm.t()
        A = torch.exp(-self.beta * (1.0 - sim_matrix))
        logits_cache = ((A @ self.W_2) / self.K_avg) * logit_scale_val
        return logits_zs + self.alpha * logits_cache

def train_tip_adapter_f(supp_f, supp_l, val_f, val_l, text_f, alpha=1.0, beta=10.0, epochs=TIP_EPOCHS, lr=0.001):
    model_f = TipAdapterF(supp_f.to(device), supp_l.to(device), text_f.to(device), alpha=alpha, beta=beta).to(device)
    optimizer = torch.optim.SGD([model_f.W_1], lr=lr, momentum=0.9, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    
    loader = DataLoader(TensorDataset(supp_f.to(device), supp_l.to(device)), batch_size=256, shuffle=True)
    history = {"loss": [], "val_acc": []}
    best_val_acc = -1.0
    best_weights = None
    
    for epoch in range(epochs):
        model_f.train()
        total_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            out = model_f(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(xb)
        history["loss"].append(total_loss / len(supp_f))
        
        if val_f is not None:
            model_f.eval()
            with torch.no_grad():
                val_out = model_f(val_f.to(device))
                val_acc = (val_out.argmax(dim=-1) == val_l.to(device)).float().mean().item() * 100
                history["val_acc"].append(val_acc)
                if val_acc >= best_val_acc:
                    best_val_acc = val_acc
                    best_weights = {k: v.clone() for k, v in model_f.state_dict().items()}
        else:
            history["val_acc"].append(None)
            
    if best_weights is not None:
        model_f.load_state_dict(best_weights)
        
    num_params = model_f.W_1.numel()
    return model_f, history, best_val_acc, num_params

print("🚀 Huấn luyện Tip-Adapter-F (Fine-Tuned) trên 9:1 Split...")
model_tip_90, hist_tip_90, val_acc_tip_90, params_tip_90 = train_tip_adapter_f(
    train_feats_90, train_lbls_90, val_feats_10, val_lbls_10, text_feats_enr, alpha=best_alpha_90, beta=best_beta_90, epochs=TIP_EPOCHS, lr=0.001
)

print("🚀 Huấn luyện Tip-Adapter-F (Fine-Tuned) trên Full Trainval...")
model_tip_full, hist_tip_full, _, params_tip_full = train_tip_adapter_f(
    train_feats_full, train_lbls_full, None, None, text_feats_enr, alpha=best_alpha_90, beta=best_beta_90, epochs=TIP_EPOCHS, lr=0.001
)

acc_tip_full_raw = (model_tip_full(test_features_raw.to(device)).argmax(dim=-1) == test_labels.to(device)).float().mean().item() * 100
acc_tip_90_tta = (model_tip_90(test_features_tta.to(device)).argmax(dim=-1) == test_labels.to(device)).float().mean().item() * 100
acc_tip_full_tta = (model_tip_full(test_features_tta.to(device)).argmax(dim=-1) == test_labels.to(device)).float().mean().item() * 100

benchmark_12_results["3. CLIP + Tip-Adapter-F (Tuned) | Raw Image (Full Trainval)"] = {
    "Method": "CLIP + Tip-Adapter-F (Tuned)", "Scenario": "Raw Image (Full Trainval)", "Val_Acc": None, "Test_Acc": acc_tip_full_raw, "Trainable_Params": params_tip_full
}
benchmark_12_results["3. CLIP + Tip-Adapter-F (Tuned) | TTA 5-view (9:1 Split)"] = {
    "Method": "CLIP + Tip-Adapter-F (Tuned)", "Scenario": "TTA 5-view (9:1 Split)", "Val_Acc": val_acc_tip_90, "Test_Acc": acc_tip_90_tta, "Trainable_Params": params_tip_90
}
benchmark_12_results["3. CLIP + Tip-Adapter-F (Tuned) | TTA 5-view (Full Trainval)"] = {
    "Method": "CLIP + Tip-Adapter-F (Tuned)", "Scenario": "TTA 5-view (Full Trainval)", "Val_Acc": None, "Test_Acc": acc_tip_full_tta, "Trainable_Params": params_tip_full
}

print("="*75)
print("🏆 KẾT QUẢ ĐÁNH GIÁ TIP-ADAPTER (FREE) & TIP-ADAPTER-F (TUNED)")
print("="*75)
print(f"  [Tip-Adapter Free]  Raw (Full): {acc_tip_free_raw:.2f}% | 9:1 TTA: {acc_tip_free_90_tta:.2f}% | Full TTA: {acc_tip_free_full_tta:.2f}%")
print(f"  [Tip-Adapter-F Tuned] Raw (Full): {acc_tip_full_raw:.2f}% | 9:1 TTA: {acc_tip_90_tta:.2f}% (Val: {val_acc_tip_90:.2f}%) | Full TTA: {acc_tip_full_tta:.2f}%")
print("="*75)
save_checkpoint('model_tip_90', model_tip_90)
save_checkpoint('model_tip_full', model_tip_full)
torch.save({'text_features': text_feats_enr.cpu(), 'support_features': train_feats_full, 'support_labels': train_lbls_full, 'classes': class_names, 'fingerprint': manifest['fingerprint'], 'alpha': best_alpha_90, 'beta': best_beta_90, 'logit_scale': logit_scale_val}, output_dir / 'tip_free_cache.pt')


## 5. PHƯƠNG PHÁP 4: CLIP + CLIP-ADAPTER (RESIDUAL FEATURE ADAPTER)

---

**CLIP-Adapter** tích hợp mạng nơ-ron truyền thẳng MLP Bottleneck thặng dư:
$$\mathbf{f}_{\text{adapted}} = \text{Normalize}\left(\alpha \cdot \text{MLP}(\mathbf{f}_{\text{img}}) + (1 - \alpha) \cdot \mathbf{f}_{\text{img}}\right)$$
$$\text{Logits} = \tau \cdot \left(\mathbf{f}_{\text{adapted}} \mathbf{W}_{\text{text}}^T\right)$$

Chúng ta đánh giá 3 kịch bản:
- **3.1. CLIP-Adapter | Raw Image (Full Trainval)**: Huấn luyện MLP trên toàn bộ 100% trainval, đánh giá trên ảnh Test chuẩn (Raw).
- **3.2. CLIP-Adapter | TTA 5-view (9:1 Split)**: Huấn luyện MLP trên 90% train (với Val 10% chọn checkpoint theo validation), đánh giá trên ảnh Test với TTA 5 góc nhìn.
- **3.3. CLIP-Adapter | TTA 5-view (Full Trainval)**: Huấn luyện MLP trên toàn bộ 100% trainval, đánh giá trên ảnh Test với TTA 5 góc nhìn.

In [ ]:
class CLIPAdapter(nn.Module):
    def __init__(self, input_dim=512, bottleneck_dim=64, alpha=0.2):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, bottleneck_dim),
            nn.ReLU(),
            nn.Linear(bottleneck_dim, input_dim)
        )
        self.alpha = alpha
        nn.init.constant_(self.mlp[2].weight, 0.0)
        nn.init.constant_(self.mlp[2].bias, 0.0)
        
    def forward(self, x):
        res = self.mlp(x)
        x_adapted = self.alpha * res + (1.0 - self.alpha) * x
        return x_adapted / x_adapted.norm(dim=-1, keepdim=True)

class AdapterClassifier(nn.Module):
    def __init__(self, adapter, text_features, logit_scale=100.0):
        super().__init__()
        self.adapter = adapter
        self.register_buffer("text_features", text_features.clone())
        self.logit_scale = logit_scale
        
    def forward(self, x):
        x_adapted = self.adapter(x)
        logits = (x_adapted @ self.text_features.t()) * self.logit_scale
        return logits

def train_clip_adapter(supp_f, supp_l, val_f, val_l, text_f, epochs=ADAPTER_EPOCHS, lr=0.001):
    adapter = CLIPAdapter(input_dim=512, bottleneck_dim=64, alpha=0.2).to(device)
    model_adapter = AdapterClassifier(adapter, text_f, logit_scale=logit_scale_val).to(device)
    
    optimizer = torch.optim.AdamW(model_adapter.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    
    loader = DataLoader(TensorDataset(supp_f.to(device), supp_l.to(device)), batch_size=256, shuffle=True)
    history = {"loss": [], "val_acc": []}
    best_val_acc = -1.0
    best_weights = None
    
    for epoch in range(epochs):
        model_adapter.train()
        total_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            out = model_adapter(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(xb)
        history["loss"].append(total_loss / len(supp_f))
        
        if val_f is not None:
            model_adapter.eval()
            with torch.no_grad():
                val_out = model_adapter(val_f.to(device))
                val_acc = (val_out.argmax(dim=-1) == val_l.to(device)).float().mean().item() * 100
                history["val_acc"].append(val_acc)
                if val_acc >= best_val_acc:
                    best_val_acc = val_acc
                    best_weights = {k: v.clone() for k, v in model_adapter.state_dict().items()}
        else:
            history["val_acc"].append(None)
            
    if best_weights is not None:
        model_adapter.load_state_dict(best_weights)
        
    num_params = sum(p.numel() for p in model_adapter.parameters() if p.requires_grad)
    return model_adapter, history, best_val_acc, num_params

print("🚀 Huấn luyện CLIP-Adapter trên 9:1 Split...")
model_ca_90, hist_ca_90, val_acc_ca_90, params_ca = train_clip_adapter(
    train_feats_90, train_lbls_90, val_feats_10, val_lbls_10, text_feats_enr, epochs=ADAPTER_EPOCHS, lr=0.001
)

print("🚀 Huấn luyện CLIP-Adapter trên Full Trainval...")
model_ca_full, hist_ca_full, _, _ = train_clip_adapter(
    train_feats_full, train_lbls_full, None, None, text_feats_enr, epochs=ADAPTER_EPOCHS, lr=0.001
)

# Đánh giá 3 kịch bản
acc_ca_full_raw = (model_ca_full(test_features_raw.to(device)).argmax(dim=-1) == test_labels.to(device)).float().mean().item() * 100
acc_ca_90_tta = (model_ca_90(test_features_tta.to(device)).argmax(dim=-1) == test_labels.to(device)).float().mean().item() * 100
acc_ca_full_tta = (model_ca_full(test_features_tta.to(device)).argmax(dim=-1) == test_labels.to(device)).float().mean().item() * 100

benchmark_12_results["3. CLIP + CLIP-Adapter | Raw Image (Full Trainval)"] = {
    "Method": "CLIP + CLIP-Adapter", "Scenario": "Raw Image (Full Trainval)", "Val_Acc": None, "Test_Acc": acc_ca_full_raw, "Trainable_Params": params_ca
}
benchmark_12_results["3. CLIP + CLIP-Adapter | TTA 5-view (9:1 Split)"] = {
    "Method": "CLIP + CLIP-Adapter", "Scenario": "TTA 5-view (9:1 Split)", "Val_Acc": val_acc_ca_90, "Test_Acc": acc_ca_90_tta, "Trainable_Params": params_ca
}
benchmark_12_results["3. CLIP + CLIP-Adapter | TTA 5-view (Full Trainval)"] = {
    "Method": "CLIP + CLIP-Adapter", "Scenario": "TTA 5-view (Full Trainval)", "Val_Acc": None, "Test_Acc": acc_ca_full_tta, "Trainable_Params": params_ca
}

print("="*75)
print("🏆 KẾT QUẢ ĐÁNH GIÁ PHƯƠNG PHÁP 3: CLIP + CLIP-ADAPTER")
print("="*75)
print(f"  3.1. Raw Image (Full Trainval):           {acc_ca_full_raw:.2f}%")
print(f"  3.2. TTA 5-view (9:1 Split):             {acc_ca_90_tta:.2f}% (Val: {val_acc_ca_90:.2f}%)")
print(f"  3.3. TTA 5-view (Full Trainval):         {acc_ca_full_tta:.2f}%")
print("="*75)

save_checkpoint('model_ca_90', model_ca_90)
save_checkpoint('model_ca_full', model_ca_full)

## 6. PHƯƠNG PHÁP 5: CLIP + PROMPT TUNING (CoOp - CONTEXT OPTIMIZATION)

---

**CoOp (Context Optimization)** tối ưu hóa $M=4$ vector ngữ cảnh liên tục $\mathbf{v}_1, \dots, \mathbf{v}_M \in \mathbb{R}^{512}$ ($2,048$ tham số) trong không gian embedding của Text Encoder:
$$\mathbf{t}^{(c)} = [\text{SOS}, \mathbf{v}_1, \mathbf{v}_2, \mathbf{v}_3, \mathbf{v}_4, \mathbf{c}_{\text{class}}, \text{EOS}]$$

Chúng ta đánh giá 3 kịch bản:
- **4.1. CoOp Prompt Tuning | Raw Image (Full Trainval)**: Huấn luyện vector ngữ cảnh trên toàn bộ 100% trainval, đánh giá trên ảnh Test chuẩn (Raw).
- **4.2. CoOp Prompt Tuning | TTA 5-view (9:1 Split)**: Huấn luyện vector ngữ cảnh trên 90% train (với Val 10% chọn checkpoint theo validation), đánh giá trên ảnh Test với TTA 5 góc nhìn.
- **4.3. CoOp Prompt Tuning | TTA 5-view (Full Trainval)**: Huấn luyện vector ngữ cảnh trên toàn bộ 100% trainval, đánh giá trên ảnh Test với TTA 5 góc nhìn.

In [ ]:
def construct_coop_input_ids(class_names, processor, num_ctx=4):
    coop_input_ids = []
    for name in class_names:
        tokens = processor.tokenizer(name, return_tensors="pt")["input_ids"][0]
        sos = tokens[:1]
        name_tokens = tokens[1:-1]
        eos = tokens[-1:]
        ctx_tokens = torch.tensor([49400 + i for i in range(num_ctx)])
        seq = torch.cat([sos, ctx_tokens, name_tokens, eos])
        padding_len = 77 - len(seq)
        if padding_len > 0:
            pad = torch.zeros(padding_len, dtype=torch.long)
            seq = torch.cat([seq, pad])
        else:
            seq = seq[:77]
            seq[-1] = eos[0]
        coop_input_ids.append(seq)
    return torch.stack(coop_input_ids)

coop_input_ids = construct_coop_input_ids(prompt_class_names, clip_processor, num_ctx=4).to(device)

class CoOpClassifier(nn.Module):
    def __init__(self, clip_model, num_classes=None, num_ctx=4):
        super().__init__()
        self.clip_model = clip_model
        self.num_ctx = num_ctx
        self.ctx = nn.Parameter(torch.randn(num_ctx, clip_model.config.text_config.hidden_size, device=device) * 0.02)

    def get_text_features(self, input_ids):
        # Scoped hook: each classifier uses its own context, including after subsequent fits.
        def replace_context(module, args, output):
            return torch.cat([output[:, :1], self.ctx.to(output.dtype).unsqueeze(0).expand(output.shape[0], -1, -1), output[:, 1 + self.num_ctx:]], dim=1)
        self.clip_model.eval()
        handle = self.clip_model.text_model.embeddings.token_embedding.register_forward_hook(replace_context)
        try:
            features = self.clip_model.get_text_features(input_ids=input_ids)
        finally:
            handle.remove()
        return F.normalize(as_feature_tensor(features), dim=-1)

    def forward(self, image_features, input_ids):
        return (image_features @ self.get_text_features(input_ids).t()) * self.clip_model.logit_scale.exp()


def train_coop(supp_f, supp_l, val_f, val_l, coop_input_ids, epochs=COOP_EPOCHS, lr=0.002):
    model_coop = CoOpClassifier(clip_model, num_classes=num_classes, num_ctx=4).to(device)
    optimizer = torch.optim.AdamW([model_coop.ctx], lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    
    loader = DataLoader(TensorDataset(supp_f.to(device), supp_l.to(device)), batch_size=256, shuffle=True)
    history = {"loss": [], "val_acc": []}
    best_val_acc = -1.0
    best_weights = None
    
    for epoch in range(epochs):
        model_coop.train()
        total_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            out = model_coop(xb, coop_input_ids)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(xb)
        history["loss"].append(total_loss / len(supp_f))
        
        if val_f is not None:
            model_coop.eval()
            with torch.no_grad():
                val_out = model_coop(val_f.to(device), coop_input_ids)
                val_acc = (val_out.argmax(dim=-1) == val_l.to(device)).float().mean().item() * 100
                history["val_acc"].append(val_acc)
                if val_acc >= best_val_acc:
                    best_val_acc = val_acc
                    best_weights = model_coop.ctx.detach().clone()
        else:
            history["val_acc"].append(None)
            
    if best_weights is not None:
        model_coop.ctx.data.copy_(best_weights)
        
    num_params = model_coop.ctx.numel()
    return model_coop, history, best_val_acc, num_params

print("🚀 Huấn luyện CoOp Prompt Tuning trên 9:1 Split...")
model_coop_90, hist_coop_90, val_acc_coop_90, params_coop = train_coop(
    train_feats_90, train_lbls_90, val_feats_10, val_lbls_10, coop_input_ids, epochs=COOP_EPOCHS, lr=0.002
)

print("🚀 Huấn luyện CoOp Prompt Tuning trên Full Trainval...")
model_coop_full, hist_coop_full, _, _ = train_coop(
    train_feats_full, train_lbls_full, None, None, coop_input_ids, epochs=COOP_EPOCHS, lr=0.002
)

# Đánh giá 3 kịch bản
acc_coop_full_raw = (model_coop_full(test_features_raw.to(device), coop_input_ids).argmax(dim=-1) == test_labels.to(device)).float().mean().item() * 100
acc_coop_90_tta = (model_coop_90(test_features_tta.to(device), coop_input_ids).argmax(dim=-1) == test_labels.to(device)).float().mean().item() * 100
acc_coop_full_tta = (model_coop_full(test_features_tta.to(device), coop_input_ids).argmax(dim=-1) == test_labels.to(device)).float().mean().item() * 100

benchmark_12_results["4. CLIP + Prompt Tuning | Raw Image (Full Trainval)"] = {
    "Method": "CLIP + Prompt Tuning", "Scenario": "Raw Image (Full Trainval)", "Val_Acc": None, "Test_Acc": acc_coop_full_raw, "Trainable_Params": params_coop
}
benchmark_12_results["4. CLIP + Prompt Tuning | TTA 5-view (9:1 Split)"] = {
    "Method": "CLIP + Prompt Tuning", "Scenario": "TTA 5-view (9:1 Split)", "Val_Acc": val_acc_coop_90, "Test_Acc": acc_coop_90_tta, "Trainable_Params": params_coop
}
benchmark_12_results["4. CLIP + Prompt Tuning | TTA 5-view (Full Trainval)"] = {
    "Method": "CLIP + Prompt Tuning", "Scenario": "TTA 5-view (Full Trainval)", "Val_Acc": None, "Test_Acc": acc_coop_full_tta, "Trainable_Params": params_coop
}

print("="*75)
print("🏆 KẾT QUẢ ĐÁNH GIÁ PHƯƠNG PHÁP 4: CLIP + PROMPT TUNING (CoOp)")
print("="*75)
print(f"  4.1. Raw Image (Full Trainval):           {acc_coop_full_raw:.2f}%")
print(f"  4.2. TTA 5-view (9:1 Split):             {acc_coop_90_tta:.2f}% (Val: {val_acc_coop_90:.2f}%)")
print(f"  4.3. TTA 5-view (Full Trainval):         {acc_coop_full_tta:.2f}%")
print("="*75)

save_checkpoint('model_coop_90', model_coop_90)
save_checkpoint('model_coop_full', model_coop_full)

## 7. ĐO ĐẠC ĐỘ TRỄ SUY DIỄN

Đo trên thiết bị thực tế, đồng bộ CUDA/MPS. Tách head-only và encoder-only; không tính đọc ảnh/tiền xử lý.


In [ ]:

predictors = {
    'CLIP (Baseline)': lambda x: x @ text_feats_enr.t() * logit_scale_val,
    'CLIP + Tip-Adapter (Free)': lambda x: run_tip_adapter_free(x, text_feats_enr, train_feats_90.to(device), train_lbls_90.to(device), best_alpha_90, best_beta_90),
    'CLIP + Tip-Adapter-F (Tuned)': model_tip_90.eval(),
    'CLIP + CLIP-Adapter': model_ca_90.eval(),
    'CLIP + Prompt Tuning': lambda x: model_coop_90(x, coop_input_ids),
}
histories = {'Tip-Adapter-F': hist_tip_90, 'CLIP-Adapter': hist_ca_90, 'CoOp': hist_coop_90}
results = benchmark_12_results
encoder = lambda x: clip_model.get_image_features(pixel_values=x)
scenario_predictors = {
    'CLIP (Baseline)': [predictors['CLIP (Baseline)']] * 3,
    'CLIP + Tip-Adapter (Free)': [
        lambda x: run_tip_adapter_free(x, text_feats_enr, train_feats_full.to(device), train_lbls_full.to(device), best_alpha_90, best_beta_90),
        predictors['CLIP + Tip-Adapter (Free)'],
        lambda x: run_tip_adapter_free(x, text_feats_enr, train_feats_full.to(device), train_lbls_full.to(device), best_alpha_90, best_beta_90)],
    'CLIP + Tip-Adapter-F (Tuned)': [model_tip_full.eval(), model_tip_90.eval(), model_tip_full.eval()],
    'CLIP + CLIP-Adapter': [model_ca_full.eval(), model_ca_90.eval(), model_ca_full.eval()],
    'CLIP + Prompt Tuning': [lambda x: model_coop_full(x, coop_input_ids), predictors['CLIP + Prompt Tuning'], lambda x: model_coop_full(x, coop_input_ids)]}
measure_scenarios(scenario_predictors)
for method, stem in [('CLIP (Baseline)', 'base'), ('CLIP + Tip-Adapter (Free)', 'tip_free'),
                     ('CLIP + Tip-Adapter-F (Tuned)', 'tip_tuned'), ('CLIP + CLIP-Adapter', 'clip_ad'),
                     ('CLIP + Prompt Tuning', 'coop')]:
    globals()[f'lat_{stem}_raw'] = scenario_latency[method][0]
    globals()[f'lat_{stem}_tta'] = scenario_latency[method][1]
    globals()[f'tp_{stem}_raw'] = 1000 / scenario_latency[method][0]
    globals()[f'tp_{stem}_tta'] = 1000 / scenario_latency[method][1]


## 8. TỔNG HỢP KẾT QUẢ & TRỰC QUAN HÓA BIỂU ĐỒ TOÀN DIỆN

---

Trực quan hóa 7 biểu đồ trọng tâm độc lập, rõ ràng và trực quan:
1. **Biểu đồ 1**: Ma trận đối sánh độ chính xác Test (%) giữa **5 phương pháp trên cả 3 kịch bản** (15 cấu hình benchmark).
2. **Biểu đồ 2**: Biểu đồ cột dọc so sánh độ chính xác Test Acc giữa các phương pháp.
3. **Biểu đồ 3**: Đường cong học tập & hội tụ (Training Loss & Validation Accuracy vs Epochs) của các phương pháp tinh chỉnh (đường màu mượt, không dùng marker).
4. **Biểu đồ 4**: Tốc độ thông lượng suy diễn thực tế (Throughput FPS) & Dung lượng tham số huấn luyện độc lập (Kịch Bản A).
5. **Biểu đồ 5**: Ma trận nhầm lẫn (Confusion Matrix) trên các giống NewDataset thú cưng của mô hình CoOp + TTA.
6. **Biểu đồ 6**: Phân tích mức tăng trưởng độ chính xác (Delta Gain) theo từng giống động vật.
7. **Biểu đồ 7**: Bản đồ đánh đổi Đa biến (Accuracy vs Latency Map & Đường biên tối ưu Pareto) - Phân định rõ Single-view và TTA 5-view.

In [ ]:

build_report(results, predictors, histories, encoder)
# Original plotting code, with measured values and automatic data limits.
_original_show = plt.show
_figure_number = [0]
def _save_original_figure(*args, **kwargs):
    _figure_number[0] += 1
    plt.savefig(output_dir / f'{_figure_number[0]:02d}_legacy_figure.png', dpi=160, bbox_inches='tight')
    _original_show(*args, **kwargs)
plt.show = _save_original_figure
try:
    with torch.no_grad():
        # ==============================================================================
        # HÌNH 1: MA TRẬN ĐỐI SÁNH KẾT QUẢ ĐỘ CHÍNH XÁC CỦA 15 CẤU HÌNH BENCHMARK
        # ==============================================================================
        scenarios = ["Raw Image (Full Trainval)", "TTA 5-view (9:1 Split)", "TTA 5-view (Full Trainval)"]
        methods = [
            "CLIP (Baseline)", 
            "CLIP + Tip-Adapter (Free)",
            "CLIP + Tip-Adapter-F (Tuned)", 
            "CLIP + CLIP-Adapter", 
            "CLIP + Prompt Tuning (CoOp)"
        ]
        
        acc_matrix = np.zeros((len(methods), len(scenarios)))
        
        for key, res in benchmark_12_results.items():
            m_name = res["Method"]
            sc_name = res["Scenario"]
            
            r_idx = None
            if "Baseline" in m_name: r_idx = 0
            elif "Tip-Adapter (Free)" in m_name: r_idx = 1
            elif "Tip-Adapter-F" in m_name or "Tip-Adapter (Tuned)" in m_name: r_idx = 2
            elif "CLIP-Adapter" in m_name: r_idx = 3
            elif "Prompt Tuning" in m_name or "CoOp" in m_name: r_idx = 4
            
            c_idx = scenarios.index(sc_name)
            if r_idx is not None:
                acc_matrix[r_idx, c_idx] = res["Test_Acc"]
        
        plt.figure(figsize=(10, 6))
        sns.heatmap(acc_matrix, annot=True, fmt=".2f", cmap="YlGnBu", xticklabels=scenarios, yticklabels=methods, cbar_kws={'label': 'Test Accuracy (%)'})
        plt.title("HÌNH 1: MA TRẬN ĐỘ CHÍNH XÁC (TEST ACCURACY %) CỦA 15 CẤU HÌNH BENCHMARK", fontsize=13, fontweight='bold', pad=15)
        plt.xlabel("Kịch Bản Tiền Xử Lý & Dữ Liệu", fontsize=11, fontweight='bold')
        plt.ylabel("Phương Pháp / Kiến Trúc", fontsize=11, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        # ==============================================================================
        # HÌNH 2: BIỂU ĐỒ CỘT DỌC SO SÁNH ĐỘ CHÍNH XÁC TEST ACCURACY GIỮA CÁC PHƯƠNG PHÁP
        # ==============================================================================
        plt.figure(figsize=(12, 6))
        x = np.arange(len(scenarios))
        width = 0.15
        
        colors = ['#5C6B73', '#D97706', '#F59E0B', '#3B82F6', '#10B981']
        
        for i in range(len(methods)):
            bars = plt.bar(x + i*width, acc_matrix[i], width, label=methods[i], color=colors[i], edgecolor='#1E293B', alpha=0.9)
            for bar in bars:
                h = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2.0, h + 0.1, f"{h:.2f}%", ha='center', va='bottom', fontsize=8, fontweight='bold', rotation=0)
        
        plt.ylabel("Độ chính xác Test Acc (%)", fontsize=11, fontweight='bold')
        plt.title("HÌNH 2: SO SÁNH ĐỘ CHÍNH XÁC TEST ACCURACY GIỮA 5 PHƯƠNG PHÁP QUA 3 KỊCH BẢN", fontsize=13, fontweight='bold', pad=15)
        plt.xticks(x + width*2, scenarios, fontsize=10, fontweight='bold')
        
        plt.legend(loc='upper left', frameon=True)
        plt.tight_layout()
        plt.show()
        
        # ==============================================================================
        # HÌNH 3: ĐƯỜNG CONG HỘI TỤ TRAIN LOSS VÀ VAL ACCURACY TRÊN 9:1 SPLIT (ĐƯỜNG MÀU MƯỢT, KHÔNG MARKER)
        # ==============================================================================
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        epochs_range = range(1, len(hist_tip_90["loss"]) + 1)
        axes[0].plot(epochs_range, hist_tip_90["loss"], label="Tip-Adapter-F (Tuned)", color='#F59E0B', linewidth=2.2)
        axes[0].plot(range(1, len(hist_ca_90["loss"]) + 1), hist_ca_90["loss"], label="CLIP-Adapter", color='#3B82F6', linewidth=2.2)
        axes[0].plot(range(1, len(hist_coop_90["loss"]) + 1), hist_coop_90["loss"], label="CoOp Prompt Tuning", color='#10B981', linewidth=2.2)
        axes[0].set_title("HÌNH 3A: ĐƯỜNG CONG TỔN THẤT (TRAINING LOSS CURVE)", fontsize=12, fontweight='bold')
        axes[0].set_xlabel("Epoch", fontsize=10, fontweight='bold')
        axes[0].set_ylabel("Cross Entropy Loss", fontsize=10, fontweight='bold')
        axes[0].legend()
        axes[0].grid(True, linestyle='--', alpha=0.5)
        
        axes[1].plot(epochs_range, hist_tip_90["val_acc"], label=f"Tip-Adapter-F (Val: {val_acc_tip_90:.2f}%)", color='#F59E0B', linewidth=2.2)
        axes[1].plot(range(1, len(hist_ca_90["loss"]) + 1), hist_ca_90["val_acc"], label=f"CLIP-Adapter (Val: {val_acc_ca_90:.2f}%)", color='#3B82F6', linewidth=2.2)
        axes[1].plot(range(1, len(hist_coop_90["loss"]) + 1), hist_coop_90["val_acc"], label=f"CoOp Prompt Tuning (Val: {val_acc_coop_90:.2f}%)", color='#10B981', linewidth=2.2)
        axes[1].set_title("HÌNH 3B: ĐƯỜNG CONG ĐỘ CHÍNH XÁC VALIDATION ACCURACY (%)", fontsize=12, fontweight='bold')
        axes[1].set_xlabel("Epoch", fontsize=10, fontweight='bold')
        axes[1].set_ylabel("Validation Accuracy (%)", fontsize=10, fontweight='bold')
        axes[1].legend()
        axes[1].grid(True, linestyle='--', alpha=0.5)
        
        plt.tight_layout()
        plt.show()
        
        # ==============================================================================
        # HÌNH 4: TỐC ĐỘ THÔNG LƯỢNG SUY DIỄN (THROUGHPUT SAMPLES/SEC) & SỐ LƯỢNG THAM SỐ (KỊCH BẢN A)
        # ==============================================================================
        method_short = [
            "CLIP (Baseline)", 
            "Tip-Adapter-F", 
            "CLIP-Adapter", 
            "CoOp Prompt Tuning"
        ]
        throughputs_raw = [tp_base_raw, tp_tip_tuned_raw, tp_clip_ad_raw, tp_coop_raw]
        params_list = [0, params_tip_full, params_ca, params_coop]
        bar_colors = ['#5C6B73', '#F59E0B', '#3B82F6', '#10B981']
        
        fig, ax_thru = plt.subplots(figsize=(12, 5))
        y_pos = np.arange(len(method_short))
        
        bars = ax_thru.barh(y_pos, throughputs_raw, color=bar_colors, edgecolor='#1E293B', alpha=0.9, height=0.55)
        ax_thru.set_yticks(y_pos)
        ax_thru.set_yticklabels(method_short, fontsize=11, fontweight='bold')
        ax_thru.set_title("HÌNH 4: TỐC ĐỘ THÔNG LƯỢNG SUY DIỄN (THROUGHPUT SAMPLES/SEC) & SỐ LƯỢNG THAM SỐ", fontsize=13, fontweight='bold', pad=15)
        ax_thru.set_xlabel("Số lượng ảnh xử lý / giây (FPS) - Càng lớn tốc độ càng cao", fontsize=11, fontweight='bold')
        
        max_tp = max(throughputs_raw)
        for bar, tp, p in zip(bars, throughputs_raw, params_list):
            w = bar.get_width()
            p_str = "0 (Zero-Shot)" if p == 0 else f"{p:,} params"
            ax_thru.text(w + max_tp * 0.02, bar.get_y() + bar.get_height()/2.0, f"{int(tp):,} FPS ({p_str})", va='center', fontsize=10, fontweight='bold', color='#1E293B')
        
        
        plt.tight_layout()
        plt.show()
        
        # ==============================================================================
        # HÌNH 5: MA TRẬN NHẦM LẪN (CONFUSION MATRIX) CỦA MÔ HÌNH TỐI ƯU
        # ==============================================================================
        best_preds = model_coop_90(test_features_tta.to(device), coop_input_ids).argmax(dim=-1).cpu()
        cm = confusion_matrix(test_labels.numpy(), best_preds.numpy())
        
        plt.figure(figsize=(18, 16))
        sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=class_names, yticklabels=class_names, cbar_kws={'label': 'Số lượng ảnh'})
        plt.title("HÌNH 5: MA TRẬN NHẦM LẪN (CONFUSION MATRIX) - COOP + TTA 5-VIEWS TRÊN NEWDATASET", fontsize=14, fontweight='bold', pad=15)
        plt.xlabel("Nhãn Dự Đoán (Predicted Breed)", fontsize=12, fontweight='bold')
        plt.ylabel("Nhãn Thực Tế (Ground Truth Breed)", fontsize=12, fontweight='bold')
        plt.xticks(rotation=90, fontsize=9)
        plt.yticks(rotation=0, fontsize=9)
        plt.tight_layout()
        plt.show()
        
        # ==============================================================================
        # HÌNH 6: PHÂN TÍCH MỨC ĐỘ CẢI THIỆN ĐỘ CHÍNH XÁC THEO TỪNG GIỐNG (DELTA GAIN)
        # ==============================================================================
        per_class_baseline = []
        per_class_best = []
        
        for c in range(num_classes):
            mask = (test_labels == c)
            acc_b = (preds_base_raw.cpu()[mask] == test_labels[mask]).float().mean().item() * 100
            acc_opt = (best_preds[mask] == test_labels[mask]).float().mean().item() * 100
            per_class_baseline.append(acc_b)
            per_class_best.append(acc_opt)
        
        per_class_delta = np.array(per_class_best) - np.array(per_class_baseline)
        top_idx = np.argsort(per_class_delta)[-10:] # Top 10 giống cải thiện mạnh nhất
        
        plt.figure(figsize=(12, 6))
        bars = plt.barh([class_names[i] for i in top_idx], per_class_delta[top_idx], color='#1ABC9C', edgecolor='#1E293B', alpha=0.9)
        plt.title("HÌNH 6: TOP 10 GIỐNG ĐỘNG VẬT ĐƯỢC CẢI THIỆN ĐỘ CHÍNH XÁC NHIỀU NHẤT (Δ ACCURACY)", fontsize=13, fontweight='bold', pad=12)
        plt.xlabel("Mức cải thiện độ chính xác (Δ % so với Baseline Zero-Shot ảnh chuẩn)", fontsize=11, fontweight='bold')
        for bar in bars:
            w = bar.get_width()
            plt.text(w + 0.3, bar.get_y() + bar.get_height()/2.0, f"{w:+.2f}%", va='center', fontsize=9.5, fontweight='bold')
        
        plt.tight_layout()
        plt.show()
        
        # ==============================================================================
        # ==============================================================================
        # ==============================================================================
        # HÌNH 7: BIỂU ĐỒ TÁN XẠ ĐÁNH GIÁ ĐÁNH ĐỔI PARETO (ACCURACY VS LATENCY TRADE-OFF)
        # ==============================================================================
        plt.figure(figsize=(12, 6.5))
        
        plot_data = [
            # Baseline
            {"name": "CLIP (Baseline Zero-Shot)", "lat": lat_base_raw, "acc": acc_base_raw, "color": "#4A5568", "marker": "o"},
            {"name": "CLIP (Baseline Zero-Shot)", "lat": lat_base_tta, "acc": acc_base_tta_90, "color": "#4A5568", "marker": "*"},
            # Tip Free
            {"name": "Tip-Adapter (Free)", "lat": lat_tip_free_raw, "acc": acc_tip_free_raw, "color": "#D97706", "marker": "o"},
            {"name": "Tip-Adapter (Free)", "lat": lat_tip_free_tta, "acc": acc_tip_free_90_tta, "color": "#D97706", "marker": "*"},
            # Tip Tuned
            {"name": "Tip-Adapter-F (Tuned)", "lat": lat_tip_tuned_raw, "acc": acc_tip_full_raw, "color": "#F59E0B", "marker": "o"},
            {"name": "Tip-Adapter-F (Tuned)", "lat": lat_tip_tuned_tta, "acc": acc_tip_90_tta, "color": "#F59E0B", "marker": "*"},
            # CLIP-Adapter
            {"name": "CLIP + CLIP-Adapter", "lat": lat_clip_ad_raw, "acc": acc_ca_full_raw, "color": "#3B82F6", "marker": "o"},
            {"name": "CLIP + CLIP-Adapter", "lat": lat_clip_ad_tta, "acc": acc_ca_90_tta, "color": "#3B82F6", "marker": "*"},
            # CoOp
            {"name": "CLIP + CoOp Prompt Tuning", "lat": lat_coop_raw, "acc": acc_coop_full_raw, "color": "#10B981", "marker": "o"},
            {"name": "CLIP + CoOp Prompt Tuning", "lat": lat_coop_tta, "acc": acc_coop_90_tta, "color": "#10B981", "marker": "*"},
        ]
        
        for d in plot_data:
            size = 180 if d["marker"] == "o" else 280
            plt.scatter(d["lat"], d["acc"], color=d["color"], marker=d["marker"], s=size, edgecolors="#1E293B", linewidth=1.5, zorder=4)
        
        pareto_x, pareto_y = pareto_frontier([(d["lat"], d["acc"]) for d in plot_data])
        plt.plot(pareto_x, pareto_y, linestyle="--", color="#10B981", linewidth=2.0, label="Đường biên tối ưu Pareto", zorder=3)
        
        plt.grid(True, linestyle="--", alpha=0.5, zorder=1)
        plt.title("HÌNH 7: PHÂN TÍCH ĐÁNH ĐỔI ĐỘ CHÍNH XÁC VS ĐỘ TRỄ SUY DIỄN (PARETO FRONTIER)", fontsize=13, fontweight='bold', pad=15)
        plt.xlabel("Độ trễ head trên cached features (ms / mẫu) - Càng nhỏ càng nhanh", fontsize=11, fontweight='bold')
        plt.ylabel("Độ chính xác trên tập Test (%) - Càng cao càng tốt", fontsize=11, fontweight='bold')
        
        
        from matplotlib.lines import Line2D
        legend_elements = [
            Line2D([0], [0], color='#10B981', lw=2, linestyle='--', label='Đường biên tối ưu Pareto'),
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#4A5568', markeredgecolor='#1E293B', markersize=10, label='Màu xám: CLIP (Baseline Zero-Shot)'),
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#D97706', markeredgecolor='#1E293B', markersize=10, label='Màu nâu cam: Tip-Adapter (Free)'),
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#F59E0B', markeredgecolor='#1E293B', markersize=10, label='Màu vàng cam: Tip-Adapter-F (Tuned)'),
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#3B82F6', markeredgecolor='#1E293B', markersize=10, label='Màu xanh dương: CLIP + CLIP-Adapter'),
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#10B981', markeredgecolor='#1E293B', markersize=10, label='Màu xanh lá: CLIP + CoOp Prompt Tuning'),
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#4A5568', markeredgecolor='#1E293B', markersize=9, label='• Hình tròn: Single-view (Raw Image)'),
            Line2D([0], [0], marker='*', color='w', markerfacecolor='#4A5568', markeredgecolor='#1E293B', markersize=13, label='* Ngôi sao: TTA 5-view (Optimal)')
        ]
        
        plt.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.02, 1.0), frameon=True, title="Chú Thích & Mô Hình", title_fontsize=10)
        plt.tight_layout()
        plt.show()
finally:
    plt.show = _original_show


## 9. BẢNG TỔNG KẾT & XUẤT ARTIFACT

Xem bảng được tạo phía trên và `benchmark_report.md`, CSV/JSON, checkpoints, split manifest, class mapping và PNG trong output. So sánh CLIP/ResNet chỉ khi fingerprint trùng khớp. Không dùng trực tiếp checkpoint mới với backend 37 lớp khi chưa cập nhật ánh xạ lớp và cấu hình inference.
